# 🏛️ TaxInspector LoRA V5 Production Training

**Mục tiêu:** Train LoRA adapter cho Qwen2.5-1.5B-Instruct trên dataset V4/V5 (130K records).

**Yêu cầu:**
- GPU T4 (free tier) hoặc tốt hơn
- Dataset đã upload lên Google Drive: `TaxInspector/agent_ultimate_dataset_v4.jsonl`

**Trước khi dùng notebook này:**
```bash
# Trên máy local, regenerate dataset (fix deprecated tools + splits):
cd e:\TaxInspector\Backend
python scripts/generate_mega_agent_dataset_v4.py
# Copy file data/agent_ultimate_dataset_v4.jsonl lên Google Drive
```

In [ ]:
!pip install -q torch transformers peft datasets accelerate bitsandbytes sentencepiece

In [ ]:
import torch, os, json, time, hashlib, re
from pathlib import Path
from collections import Counter

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('VRAM:', f'{torch.cuda.get_device_properties(0).total_mem / 1e9:.1f}GB' if torch.cuda.is_available() else 'N/A')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/TaxInspector')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = DRIVE_DIR / 'agent_ultimate_dataset_v4.jsonl'
OUTPUT_DIR = Path('/content/tax_agent_lora_v5_production')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_PATH.exists(), f'Dataset not found! Upload to {DATASET_PATH}'
print(f'Dataset: {DATASET_PATH.stat().st_size / 1e6:.1f}MB')

## 📋 Validate Dataset

In [ ]:
CANONICAL_TOOLS = frozenset({
    'knowledge_search', 'company_risk_lookup', 'delinquency_check',
    'invoice_risk_scan', 'vat_refund_risk', 'gnn_analysis',
    'motif_detection', 'ring_scoring', 'ownership_analysis',
    'temporal_delinquency_deep', 'hetero_gnn_risk', 'vae_anomaly_scan',
    'causal_uplift_recommend', 'top_n_risky_companies', 'company_name_search',
    'nlp_red_flag_scan', 'revenue_forecast', 'entity_resolution_check',
    'ocr_document_process', 'macro_forecast',
})
DEPRECATED = {'gnn_vat_fraud':'gnn_analysis','run_hetero_gnn':'hetero_gnn_risk',
    'run_vae_anomaly':'vae_anomaly_scan','predict_delinquency':'temporal_delinquency_deep',
    'causal_uplift_action':'causal_uplift_recommend','query_legal_graphrag':'knowledge_search',
    'run_macro_simulation':'macro_forecast'}

records = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): records.append(json.loads(line))
print(f'Total: {len(records)}')

deprecated_count = invalid_count = 0
for r in records:
    for msg in r.get('messages', r.get('conversations', [])):
        content = str(msg.get('content', msg.get('value', '')))
        m = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', content, re.DOTALL)
        if not m: continue
        try:
            name = json.loads(m.group(1)).get('name','')
            if name in DEPRECATED: deprecated_count += 1
            elif name not in CANONICAL_TOOLS: invalid_count += 1
        except: invalid_count += 1

splits = Counter(r.get('metadata',{}).get('split','?') for r in records)
print(f'Splits: {dict(splits)}')
print(f'Deprecated: {deprecated_count}, Invalid: {invalid_count}')
if deprecated_count > 0:
    print('⚠️ Regenerate dataset trước khi train!')
else:
    print('✅ Dataset OK')

## 🔧 Prepare Training Data

In [ ]:
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoTokenizer

BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ = 1024

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def to_text(r):
    msgs = r.get('messages')
    if msgs:
        parts = []
        for m in msgs:
            role, content = m.get('role',''), m.get('content','')
            if role == 'system': parts.append(f'[SYSTEM]\n{content}\n[/SYSTEM]')
            elif role == 'user': parts.append(f'[USER]\n{content}\n[/USER]')
            elif role == 'assistant': parts.append(f'[ASSISTANT]\n{content}')
            elif role == 'tool': parts.append(f'[TOOL_RESULT]\n{content}\n[/TOOL_RESULT]')
        return '\n'.join(parts)
    c = r.get('conversations',[])
    if len(c)>=3: return f"[SYSTEM]\n{c[0].get('value','')}\n[/SYSTEM]\n[USER]\n{c[1].get('value','')}\n[/USER]\n[ASSISTANT]\n{c[2].get('value','')}"
    return None

train_r = [r for r in records if r.get('metadata',{}).get('split','train') == 'train']
eval_r = [r for r in records if r.get('metadata',{}).get('split') in ('dev','eval','validation')]
if not eval_r:
    s = int(len(records)*0.9)
    train_r, eval_r = records[:s], records[s:]
print(f'Train: {len(train_r)}, Eval: {len(eval_r)}')

train_texts = [t for t in (to_text(r) for r in train_r) if t][:50000]
eval_texts = [t for t in (to_text(r) for r in eval_r) if t][:2000]
print(f'Train texts: {len(train_texts)}, Eval: {len(eval_texts)}')

print('Tokenizing...')
train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_SEQ, return_tensors='pt')
train_enc['labels'] = train_enc['input_ids'].clone()
eval_enc = tokenizer(eval_texts, truncation=True, padding=True, max_length=MAX_SEQ, return_tensors='pt')
eval_enc['labels'] = eval_enc['input_ids'].clone()

class TokDS(TorchDataset):
    def __init__(s, e): s.ids,s.mask,s.lab = e['input_ids'],e['attention_mask'],e['labels']
    def __len__(s): return len(s.ids)
    def __getitem__(s,i): return {'input_ids':s.ids[i],'attention_mask':s.mask[i],'labels':s.lab[i]}

train_ds, eval_ds = TokDS(train_enc), TokDS(eval_enc)
print(f'✅ Ready: train={len(train_ds)}, eval={len(eval_ds)}')

## 🚀 Train LoRA

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True, torch_dtype=torch.float16, device_map='auto')
lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj'])
model = get_peft_model(model, lora_cfg)
tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
tt = sum(p.numel() for p in model.parameters())
print(f'Trainable: {tp:,}/{tt:,} ({tp/tt*100:.2f}%)')

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=3,
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=2e-4,
    warmup_ratio=0.1, weight_decay=0.01,
    save_strategy='epoch', eval_strategy='epoch',
    logging_steps=50, gradient_checkpointing=True,
    fp16=True, report_to='none', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds)

print('🚀 Starting training...')
t0 = time.time()
trainer.train()
print(f'\n✅ Done in {(time.time()-t0)/60:.1f} minutes')
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

## 📊 Evaluate Tool-Call Accuracy

In [ ]:
test_r = [r for r in records if r.get('metadata',{}).get('split')=='test']
if not test_r: test_r = eval_r[:200]
print(f'Testing on {min(200,len(test_r))} records...')
model.eval()
ok = total = 0
for i, rec in enumerate(test_r[:200]):
    exp = rec.get('metadata',{}).get('expected_tool')
    if not exp: continue
    user = next((m['content'] for m in rec.get('messages',[]) if m.get('role')=='user'), None)
    if not user: continue
    msgs = [{'role':'system','content':rec['messages'][0]['content']},{'role':'user','content':user}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ)
    inp = {k:v.to(model.device) for k,v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=256, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', resp, re.DOTALL)
    if m:
        try:
            pred = json.loads(m.group(1)).get('name','')
            pred = DEPRECATED.get(pred, pred)
            exp_c = DEPRECATED.get(exp, exp)
            total += 1
            if pred == exp_c: ok += 1
        except: total += 1
    if (i+1)%50==0: print(f'  {i+1}/200...')
acc = ok/max(1,total)
print(f'\n📊 Tool accuracy: {ok}/{total} = {acc:.1%} (target ≥85%)')
Path(OUTPUT_DIR/'eval_report.json').write_text(json.dumps({'accuracy':round(acc,4),'correct':ok,'total':total,'timestamp':time.strftime('%Y-%m-%d %H:%M')},indent=2))

## 💾 Export to Google Drive

In [ ]:
import shutil
DRIVE_OUT = DRIVE_DIR / 'tax_agent_lora_v5_production'
if DRIVE_OUT.exists(): shutil.rmtree(DRIVE_OUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUT)
print(f'✅ Saved to {DRIVE_OUT}')
for f in sorted(DRIVE_OUT.glob('*')):
    if f.is_file():
        h = hashlib.sha256(f.read_bytes()).hexdigest().upper()[:16]
        print(f'  {f.name}: {h}... ({f.stat().st_size:,}B)')
print(f'''
{'='*50}
  Bước tiếp:
  1. Download thư mục từ Drive
  2. Copy vào e:\\TaxInspector\\Backend\\tax_agent\\
  3. Test: python test_agent_v5_local.py
{'='*50}
''')